In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import os
from pathlib import Path
from tqdm import tqdm
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from transformers import ViTModel, ViTFeatureExtractor
import torch.nn.functional as F
import math
import matplotlib.pyplot as plt
import cv2
from torch.cuda.amp import autocast, GradScaler
from lime import lime_image
from skimage.segmentation import mark_boundaries

# Set paths
train_path = "/kaggle/input/faceforensics/FaceForensic/train"
val_path = "/kaggle/input/faceforensics/FaceForensic/validation"
test_path = "/kaggle/input/faceforensics/FaceForensic/test"

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Clear cache
torch.cuda.empty_cache()

# ============ DIFFUSION CONFIGURATION ============

class DiffusionConfig:
    num_timesteps = 1000
    beta_start = 0.0001
    beta_end = 0.02
    reverse_chain = [800, 600, 400, 200, 0]

def get_diffusion_schedule(num_timesteps, beta_start, beta_end):
    betas = torch.linspace(beta_start, beta_end, num_timesteps)
    alphas = 1.0 - betas
    alphas_cumprod = torch.cumprod(alphas, dim=0)
    alphas_cumprod_prev = F.pad(alphas_cumprod[:-1], (1, 0), value=1.0)
    
    sqrt_alphas_cumprod = torch.sqrt(alphas_cumprod)
    sqrt_one_minus_alphas_cumprod = torch.sqrt(1.0 - alphas_cumprod)
    posterior_variance = betas * (1.0 - alphas_cumprod_prev) / (1.0 - alphas_cumprod)
    
    return {
        'betas': betas,
        'alphas': alphas,
        'alphas_cumprod': alphas_cumprod,
        'alphas_cumprod_prev': alphas_cumprod_prev,
        'sqrt_alphas_cumprod': sqrt_alphas_cumprod,
        'sqrt_one_minus_alphas_cumprod': sqrt_one_minus_alphas_cumprod,
        'posterior_variance': posterior_variance
    }

config = DiffusionConfig()
diffusion_params = get_diffusion_schedule(config.num_timesteps, config.beta_start, config.beta_end)

for key in diffusion_params:
    diffusion_params[key] = diffusion_params[key].to(device)

def forward_diffusion(x_0, t, params):
    batch_size = x_0.shape[0]
    sqrt_alpha_cumprod = params['sqrt_alphas_cumprod'][t].view(batch_size, 1, 1, 1)
    sqrt_one_minus_alpha_cumprod = params['sqrt_one_minus_alphas_cumprod'][t].view(batch_size, 1, 1, 1)
    noise = torch.randn_like(x_0)
    x_t = sqrt_alpha_cumprod * x_0 + sqrt_one_minus_alpha_cumprod * noise
    return x_t, noise

# ============ MEMORY-OPTIMIZED SCORE NETWORK ============

class ScoreNetwork(nn.Module):
    def __init__(self, in_channels=3, base_channels=64):
        super().__init__()
        
        self.time_embed_dim = 128
        self.time_mlp = nn.Sequential(
            nn.Linear(self.time_embed_dim, base_channels * 4),
            nn.SiLU(),
            nn.Linear(base_channels * 4, base_channels * 4)
        )
        
        self.enc1 = self._make_block(in_channels, base_channels)
        self.enc2 = self._make_block(base_channels, base_channels * 2, downsample=True)
        self.enc3 = self._make_block(base_channels * 2, base_channels * 4, downsample=True)
        
        self.bottleneck = self._make_block(base_channels * 4, base_channels * 4)
        
        self.dec3 = self._make_block(base_channels * 8, base_channels * 2, upsample=True)
        self.dec2 = self._make_block(base_channels * 4, base_channels, upsample=True)
        self.dec1 = self._make_block(base_channels * 2, base_channels)
        
        self.out = nn.Conv2d(base_channels, in_channels, 3, padding=1)
        
    def _make_block(self, in_ch, out_ch, downsample=False, upsample=False):
        layers = []
        if downsample:
            layers.append(nn.Conv2d(in_ch, out_ch, 3, stride=2, padding=1))
        elif upsample:
            layers.append(nn.ConvTranspose2d(in_ch, out_ch, 4, stride=2, padding=1))
        else:
            layers.append(nn.Conv2d(in_ch, out_ch, 3, padding=1))
        
        layers.extend([
            nn.GroupNorm(8, out_ch),
            nn.SiLU(),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.GroupNorm(8, out_ch),
            nn.SiLU()
        ])
        return nn.Sequential(*layers)
    
    def get_timestep_embedding(self, timesteps):
        half_dim = self.time_embed_dim // 2
        emb = math.log(10000) / (half_dim - 1)
        emb = torch.exp(torch.arange(half_dim, device=timesteps.device) * -emb)
        emb = timesteps[:, None].float() * emb[None, :]
        emb = torch.cat([torch.sin(emb), torch.cos(emb)], dim=-1)
        return emb
    
    def forward(self, x_t, t):
        t_emb = self.get_timestep_embedding(t)
        t_emb = self.time_mlp(t_emb)
        
        e1 = self.enc1(x_t)
        e2 = self.enc2(e1)
        e3 = self.enc3(e2)
        
        b = self.bottleneck(e3)
        b = b + t_emb[:, :, None, None]
        
        d3 = self.dec3(torch.cat([b, e3], dim=1))
        d2 = self.dec2(torch.cat([d3, e2], dim=1))
        d1 = self.dec1(torch.cat([d2, e1], dim=1))
        
        noise_pred = self.out(d1)
        return noise_pred

def reverse_diffusion_step(x_t, t, t_prev, noise_pred, params):
    batch_size = x_t.shape[0]
    alpha_t = params['alphas_cumprod'][t].view(batch_size, 1, 1, 1)
    is_final_step = (t_prev == 0).all().item()
    
    if is_final_step:
        alpha_t_prev = torch.ones_like(alpha_t)
    else:
        alpha_t_prev = params['alphas_cumprod'][t_prev].view(batch_size, 1, 1, 1)
    
    sqrt_alpha_t = torch.sqrt(alpha_t)
    sqrt_one_minus_alpha_t = torch.sqrt(1.0 - alpha_t)
    
    x_0_pred = (x_t - sqrt_one_minus_alpha_t * noise_pred) / sqrt_alpha_t
    x_0_pred = torch.clamp(x_0_pred, -1.0, 1.0)
    
    if not is_final_step:
        sqrt_alpha_t_prev = torch.sqrt(alpha_t_prev)
        sqrt_one_minus_alpha_t_prev = torch.sqrt(1.0 - alpha_t_prev)
        x_t_prev = sqrt_alpha_t_prev * x_0_pred + sqrt_one_minus_alpha_t_prev * noise_pred
    else:
        x_t_prev = x_0_pred
    
    return x_t_prev, x_0_pred

# ============ SPATIALLY-CONDITIONED SCORE ANALYZER ============

class SpatiallyConditionedScoreAnalyzer(nn.Module):
    def __init__(self, vit_hidden_size=768):
        super().__init__()
        
        self.diff_projection = nn.Sequential(
            nn.Conv2d(12, 64, 3, padding=1),
            nn.ReLU(),
            nn.Conv2d(64, 128, 3, stride=2, padding=1),
            nn.ReLU(),
            nn.Conv2d(128, 256, 3, stride=2, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d(7),
            nn.Flatten(start_dim=2)
        )
        
        self.query_proj = nn.Linear(256, vit_hidden_size)
        
        self.cross_attention = nn.MultiheadAttention(
            embed_dim=vit_hidden_size,
            num_heads=12,
            batch_first=True
        )
        
        self.post_attention = nn.Sequential(
            nn.Linear(vit_hidden_size, 384),
            nn.GELU(),
            nn.Dropout(0.2)
        )
        
    def forward(self, score_differences, vit_patch_embeddings):
        temporal_diffs = torch.cat(score_differences, dim=1)
        diff_features = self.diff_projection(temporal_diffs)
        diff_features = diff_features.transpose(1, 2)
        diff_queries = self.query_proj(diff_features)
        
        attended_features, attention_weights = self.cross_attention(
            diff_queries,
            vit_patch_embeddings,
            vit_patch_embeddings
        )
        
        global_features = torch.mean(attended_features, dim=1)
        conditioned_features = self.post_attention(global_features)
        
        return conditioned_features, attention_weights

# ============ MAIN MODEL ============

class SpatiallyConditionedScoreViT(nn.Module):
    def __init__(self, model_name, num_classes):
        super().__init__()
        
        self.vit = ViTModel.from_pretrained(model_name)
        hidden_size = self.vit.config.hidden_size
        
        self.score_network = ScoreNetwork(in_channels=3, base_channels=64)
        
        self.spatially_conditioned_analyzer = SpatiallyConditionedScoreAnalyzer(
            vit_hidden_size=hidden_size
        )
        
        combined_dim = hidden_size + 384
        
        self.feature_fusion = nn.Sequential(
            nn.LayerNorm(combined_dim),
            nn.Linear(combined_dim, 512),
            nn.GELU(),
            nn.Dropout(0.3)
        )
        
        self.classifier = nn.Sequential(
            nn.Linear(512, 256),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(256, num_classes)
        )
        
        self.activations = {}
        self.gradients = {}
        
    def save_activation(self, name):
        def hook(module, input, output):
            self.activations[name] = output.detach()
        return hook
    
    def save_gradient(self, name):
        def hook(module, grad_input, grad_output):
            self.gradients[name] = grad_output[0].detach()
        return hook
        
    def iterative_reverse_diffusion(self, x_T, reverse_chain, params):
        x_t = x_T
        score_predictions = []
        
        for i in range(len(reverse_chain) - 1):
            t_current = reverse_chain[i]
            t_next = reverse_chain[i + 1]
            
            t_tensor = torch.full((x_t.shape[0],), t_current, device=x_t.device, dtype=torch.long)
            t_next_tensor = torch.full((x_t.shape[0],), t_next, device=x_t.device, dtype=torch.long)
            
            noise_pred = self.score_network(x_t, t_tensor)
            score_predictions.append(noise_pred)
            
            x_t, _ = reverse_diffusion_step(x_t, t_tensor, t_next_tensor, noise_pred, params)
        
        t_final = torch.full((x_t.shape[0],), 0, device=x_t.device, dtype=torch.long)
        noise_final = self.score_network(x_t, t_final)
        score_predictions.append(noise_final)
        
        return score_predictions, x_t
    
    def forward(self, pixel_values, reverse_chain, params):
        batch_size = pixel_values.shape[0]
        x_0 = pixel_values * 2.0 - 1.0
        
        t_start = reverse_chain[0]
        t_start_tensor = torch.full((batch_size,), t_start, device=pixel_values.device, dtype=torch.long)
        
        x_T, _ = forward_diffusion(x_0, t_start_tensor, params)
        score_predictions, x_final = self.iterative_reverse_diffusion(x_T, reverse_chain, params)
        
        x_final_vit = (x_final + 1.0) / 2.0
        vit_outputs = self.vit(pixel_values=x_final_vit)
        
        patch_embeddings = vit_outputs.last_hidden_state[:, 1:, :]
        cls_token = vit_outputs.last_hidden_state[:, 0, :]
        
        score_differences = []
        for i in range(len(score_predictions) - 1):
            diff = torch.abs(score_predictions[i] - score_predictions[i+1])
            score_differences.append(diff)
        
        conditioned_score_features, attention_weights = self.spatially_conditioned_analyzer(
            score_differences, 
            patch_embeddings
        )
        
        combined = torch.cat([cls_token, conditioned_score_features], dim=1)
        fused = self.feature_fusion(combined)
        logits = self.classifier(fused)
        
        return logits

# ============ DATASET ============

class DeepfakeDataset(Dataset):
    def __init__(self, data_dir, feature_extractor):
        self.data_dir = data_dir
        self.feature_extractor = feature_extractor
        self.image_paths = []
        self.labels = []
        
        for img_name in os.listdir(data_dir):
            if img_name.endswith('.png'):
                img_path = os.path.join(data_dir, img_name)
                self.image_paths.append(img_path)
                
                if img_name.startswith('real_'):
                    self.labels.append(1)
                elif img_name.startswith('fake_'):
                    self.labels.append(0)
        
        print(f"Loaded {len(self.image_paths)} images from {data_dir}")
        print(f"Real: {sum(self.labels)}, Fake: {len(self.labels) - sum(self.labels)}")
    
    def __len__(self):
        return len(self.image_paths)
    
    def __getitem__(self, idx):
        image = Image.open(self.image_paths[idx]).convert('RGB')
        inputs = self.feature_extractor(images=image, return_tensors="pt")
        return inputs['pixel_values'].squeeze(0), self.labels[idx], self.image_paths[idx]

# ============ TRAINING SETUP ============

batch_size = 4
learning_rate = 3e-6
num_epochs = 1
num_classes = 2

model_name = "google/vit-base-patch16-224"
feature_extractor = ViTFeatureExtractor.from_pretrained(model_name)

train_dataset = DeepfakeDataset(train_path, feature_extractor)
val_dataset = DeepfakeDataset(val_path, feature_extractor)
test_dataset = DeepfakeDataset(test_path, feature_extractor)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)

model = SpatiallyConditionedScoreViT(model_name=model_name, num_classes=num_classes)
model = model.to(device)

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=0.01)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)

scaler = GradScaler()

# ============ TRAINING FUNCTIONS ============

def train_epoch(model, dataloader, criterion, optimizer, device, reverse_chain, params, scaler):
    model.train()
    running_loss = 0.0
    all_preds = []
    all_labels = []
    
    for images, labels, _ in tqdm(dataloader, desc="Training"):
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        
        optimizer.zero_grad()
        
        with autocast():
            logits = model(images, reverse_chain, params)
            loss = criterion(logits, labels)
        
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        
        running_loss += loss.item()
        preds = torch.argmax(logits, dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        
        if len(all_preds) % 100 == 0:
            torch.cuda.empty_cache()
    
    return running_loss / len(dataloader), accuracy_score(all_labels, all_preds)

def validate(model, dataloader, criterion, device, reverse_chain, params):
    model.eval()
    running_loss = 0.0
    all_preds = []
    all_labels = []
    all_paths = []
    
    with torch.no_grad():
        for images, labels, paths in tqdm(dataloader, desc="Validation"):
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            
            with autocast():
                logits = model(images, reverse_chain, params)
                loss = criterion(logits, labels)
            
            running_loss += loss.item()
            preds = torch.argmax(logits, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_paths.extend(paths)
    
    return running_loss / len(dataloader), accuracy_score(all_labels, all_preds), all_preds, all_labels, all_paths

# ============ TRAINING LOOP ============

best_val_acc = 0.0

print("\n" + "="*85)
print("SPATIALLY-CONDITIONED SCORE TRAJECTORY ANALYSIS (MEMORY OPTIMIZED)")
print("="*85 + "\n")

for epoch in range(num_epochs):
    print(f"\nEpoch {epoch+1}/{num_epochs}")
    print("-" * 55)
    
    train_loss, train_acc = train_epoch(
        model, train_loader, criterion, optimizer, device,
        config.reverse_chain, diffusion_params, scaler
    )
    
    val_loss, val_acc, _, _, _ = validate(
        model, val_loader, criterion, device,
        config.reverse_chain, diffusion_params
    )
    
    scheduler.step()
    
    print(f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}")
    print(f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")
    print(f"LR: {scheduler.get_last_lr()[0]:.7f}")
    
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), 'best_spatially_conditioned_model.pth')
        print(f"✓ Best model saved! (Val Acc: {val_acc:.4f})")
    
    torch.cuda.empty_cache()

print("\n" + "="*85)
print("Training Completed!")
print("="*85 + "\n")

# ============ TEST EVALUATION ============

model.load_state_dict(torch.load('best_spatially_conditioned_model.pth'))

test_loss, test_acc, test_preds, test_labels, test_paths = validate(
    model, test_loader, criterion, device,
    config.reverse_chain, diffusion_params
)

precision = precision_score(test_labels, test_preds, average='binary')
recall = recall_score(test_labels, test_preds, average='binary')
f1 = f1_score(test_labels, test_preds, average='binary')
cm = confusion_matrix(test_labels, test_preds)

print("\n" + "="*85)
print("FINAL TEST RESULTS")
print("="*85)
print(f"Test Accuracy: {test_acc:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1-Score: {f1:.4f}")
print("\nConfusion Matrix:")
print(f"              Fake    Real")
print(f"Fake   {cm[0][0]:4d}    {cm[0][1]:4d}")
print(f"Real   {cm[1][0]:4d}    {cm[1][1]:4d}")
print("="*85 + "\n")

# ============ ADVANCED VISUALIZATION FUNCTIONS ============

def get_activation_map(model, image, reverse_chain, params):
    """Generate activation map from ViT encoder last layer"""
    model.eval()
    
    activations = []
    
    def forward_hook(module, input, output):
        activations.append(output)
    
    handle = model.vit.encoder.layer[-1].register_forward_hook(forward_hook)
    
    image = image.unsqueeze(0).to(device)
    with torch.no_grad():
        _ = model(image, reverse_chain, params)
    
    handle.remove()
    
    # Get activation from last encoder layer
    acts = activations[0][:, 1:, :]  # Remove CLS token
    activation_map = torch.mean(acts, dim=-1).squeeze()
    
    # Reshape to spatial dimensions
    size = int(np.sqrt(activation_map.shape[0]))
    activation_map = activation_map.view(size, size)
    activation_map = activation_map / torch.max(activation_map)
    
    return activation_map.cpu().numpy()


def get_gradcam(model, image, target_class, reverse_chain, params):
    """Generate GradCAM heatmap"""
    model.eval()
    
    activations = []
    gradients = []
    
    def forward_hook(module, input, output):
        activations.append(output)
    
    def backward_hook(module, grad_input, grad_output):
        gradients.append(grad_output[0])
    
    handle_forward = model.vit.encoder.layer[-1].register_forward_hook(forward_hook)
    handle_backward = model.vit.encoder.layer[-1].register_backward_hook(backward_hook)
    
    image = image.unsqueeze(0).to(device)
    image.requires_grad = True
    
    output = model(image, reverse_chain, params)
    
    model.zero_grad()
    class_score = output[0, target_class]
    class_score.backward()
    
    acts = activations[0][:, 1:, :]  # Remove CLS token
    grads = gradients[0][:, 1:, :]
    
    # GradCAM computation
    pooled_grads = torch.mean(grads, dim=[0, 1])
    for i in range(acts.shape[-1]):
        acts[:, :, i] *= pooled_grads[i]
    
    heatmap = torch.mean(acts, dim=-1).squeeze()
    heatmap = F.relu(heatmap)
    heatmap = heatmap / (torch.max(heatmap) + 1e-8)
    
    handle_forward.remove()
    handle_backward.remove()
    
    size = int(np.sqrt(heatmap.shape[0]))
    heatmap = heatmap.view(size, size)
    
    return heatmap.cpu().detach().numpy()


def get_lime_explanation(model, image_path, reverse_chain, params, num_samples=500):
    """Generate LIME explanation"""
    
    def predict_fn(images):
        """Prediction function for LIME"""
        batch_preds = []
        model.eval()
        
        with torch.no_grad():
            for img in images:
                img_pil = Image.fromarray(img.astype('uint8'))
                inputs = feature_extractor(images=img_pil, return_tensors="pt")
                img_tensor = inputs['pixel_values'].to(device)
                
                logits = model(img_tensor, reverse_chain, params)
                probs = F.softmax(logits, dim=1)
                batch_preds.append(probs.cpu().numpy()[0])
        
        return np.array(batch_preds)
    
    # Load image
    img_pil = Image.open(image_path).convert('RGB')
    img_np = np.array(img_pil)
    
    # Create LIME explainer
    explainer = lime_image.LimeImageExplainer()
    
    # Generate explanation
    explanation = explainer.explain_instance(
        img_np,
        predict_fn,
        top_labels=2,
        hide_color=0,
        num_samples=num_samples,
        batch_size=10
    )
    
    # Get mask for top prediction
    temp, mask = explanation.get_image_and_mask(
        explanation.top_labels[0],
        positive_only=True,
        num_features=10,
        hide_rest=False
    )
    
    return mask


def get_denoised_image(model, image, reverse_chain, params):
    """Get final denoised image from reverse diffusion"""
    model.eval()
    
    with torch.no_grad():
        image = image.unsqueeze(0).to(device)
        x_0 = image * 2.0 - 1.0
        
        t_start = reverse_chain[0]
        t_start_tensor = torch.full((1,), t_start, device=device, dtype=torch.long)
        x_T, _ = forward_diffusion(x_0, t_start_tensor, params)
        
        _, x_final = model.iterative_reverse_diffusion(x_T, reverse_chain, params)
    
    x_final_display = ((x_final.squeeze(0).cpu().permute(1, 2, 0).numpy() + 1) / 2 * 255).astype(np.uint8)
    
    return x_final_display


def visualize_comprehensive_horizontal(model, image_path, label, pred, reverse_chain, params, save_path):
    """
    Create comprehensive horizontal visualization for research paper
    Includes: Original, Activation Map, GradCAM, LIME, Denoised Image
    """
    
    print(f"Processing: {os.path.basename(image_path)}")
    
    # Load image
    img_pil = Image.open(image_path).convert('RGB')
    img_np = np.array(img_pil)
    
    # Preprocess
    inputs = feature_extractor(images=img_pil, return_tensors="pt")
    image_tensor = inputs['pixel_values'].squeeze(0)
    
    # Generate all visualizations
    print("  - Generating Activation Map...")
    try:
        activation_map = get_activation_map(model, image_tensor, reverse_chain, params)
        activation_map_resized = cv2.resize(activation_map, (img_np.shape[1], img_np.shape[0]))
    except Exception as e:
        print(f"    Warning: Activation map failed - {e}")
        activation_map_resized = np.zeros((img_np.shape[0], img_np.shape[1]))
    
    print("  - Generating GradCAM...")
    try:
        gradcam_heatmap = get_gradcam(model, image_tensor, pred, reverse_chain, params)
        gradcam_heatmap_resized = cv2.resize(gradcam_heatmap, (img_np.shape[1], img_np.shape[0]))
    except Exception as e:
        print(f"    Warning: GradCAM failed - {e}")
        gradcam_heatmap_resized = np.zeros((img_np.shape[0], img_np.shape[1]))
    
    print("  - Generating LIME explanation...")
    try:
        lime_mask = get_lime_explanation(model, image_path, reverse_chain, params, num_samples=300)
    except Exception as e:
        print(f"    Warning: LIME failed - {e}")
        lime_mask = np.zeros((img_np.shape[0], img_np.shape[1]))
    
    print("  - Generating Denoised Image...")
    try:
        denoised_image = get_denoised_image(model, image_tensor, reverse_chain, params)
    except Exception as e:
        print(f"    Warning: Denoised image failed - {e}")
        denoised_image = img_np
    
    # Create horizontal visualization (1 row x 5 columns)
    fig, axes = plt.subplots(1, 5, figsize=(20, 4))
    
    # 1. Original Image
    axes[0].imshow(img_np)
    axes[0].set_title(f'Original\nTrue: {"Real" if label == 1 else "Fake"}\nPred: {"Real" if pred == 1 else "Fake"}', 
                      fontsize=10, fontweight='bold')
    axes[0].axis('off')
    
    # 2. Activation Map
    axes[1].imshow(img_np)
    axes[1].imshow(activation_map_resized, cmap='hot', alpha=0.6)
    axes[1].set_title('Activation Map\n(ViT Last Layer)', fontsize=10, fontweight='bold')
    axes[1].axis('off')
    
    # 3. GradCAM
    axes[2].imshow(img_np)
    axes[2].imshow(gradcam_heatmap_resized, cmap='jet', alpha=0.5)
    axes[2].set_title('GradCAM', fontsize=10, fontweight='bold')
    axes[2].axis('off')
    
    # 4. LIME
    axes[3].imshow(mark_boundaries(img_np / 255.0, lime_mask))
    axes[3].set_title('LIME\n(Superpixel Explanation)', fontsize=10, fontweight='bold')
    axes[3].axis('off')
    
    # 5. Denoised Image
    axes[4].imshow(denoised_image)
    axes[4].set_title('Denoised Image\n(Reverse Diffusion)', fontsize=10, fontweight='bold')
    axes[4].axis('off')
    
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight', facecolor='white')
    plt.close()
    
    print(f"  ✓ Saved: {save_path}\n")


# ============ GENERATE COMPREHENSIVE VISUALIZATIONS ============

print("\n" + "="*85)
print("GENERATING COMPREHENSIVE VISUALIZATIONS FOR RESEARCH PAPER")
print("="*85 + "\n")
print("Visualizations include:")
print("  1. Original Image")
print("  2. Activation Map (ViT Last Layer)")
print("  3. GradCAM")
print("  4. LIME (Superpixel Explanation)")
print("  5. Denoised Image (Reverse Diffusion)")
print("\nArrangement: Horizontal (1 row × 5 columns) for paper space efficiency")
print("="*85 + "\n")

# Create output directory
os.makedirs('research_visualizations', exist_ok=True)

# Select samples to visualize
real_indices = [i for i, label in enumerate(test_labels) if label == 1]
fake_indices = [i for i, label in enumerate(test_labels) if label == 0]

# Number of samples to visualize
num_samples = 3

print(f"Generating visualizations for {num_samples} REAL samples...")
print("-" * 85)
for i, idx in enumerate(real_indices[:num_samples]):
    visualize_comprehensive_horizontal(
        model, 
        test_paths[idx], 
        test_labels[idx], 
        test_preds[idx],
        config.reverse_chain, 
        diffusion_params,
        f'research_visualizations/real_sample_{i+1}.png'
    )

print(f"\nGenerating visualizations for {num_samples} FAKE samples...")
print("-" * 85)
for i, idx in enumerate(fake_indices[:num_samples]):
    visualize_comprehensive_horizontal(
        model, 
        test_paths[idx], 
        test_labels[idx], 
        test_preds[idx],
        config.reverse_chain, 
        diffusion_params,
        f'research_visualizations/fake_sample_{i+1}.png'
    )

print("\n" + "="*85)
print("✓ ALL VISUALIZATIONS GENERATED SUCCESSFULLY!")
print("="*85)
print(f"\nOutput directory: research_visualizations/")
print(f"Total files created: {num_samples * 2}")
print(f"\nFiles:")
for i in range(1, num_samples + 1):
    print(f"  - real_sample_{i}.png")
for i in range(1, num_samples + 1):
    print(f"  - fake_sample_{i}.png")
print("\nEach visualization shows 5 methods horizontally for easy paper inclusion.")
print("="*85 + "\n")

2026-04-04 05:59:51.187482: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1775282391.210206     237 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1775282391.216685     237 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1775282391.234214     237 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775282391.234240     237 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775282391.234242     237 computation_placer.cc:177] computation placer alr

Using device: cuda


/usr/local/lib/python3.12/dist-packages/transformers/models/vit/feature_extraction_vit.py:30: FutureWarning: The class ViTFeatureExtractor is deprecated and will be removed in version 5 of Transformers. Please use ViTImageProcessor instead.
  warnings.warn(


Loaded 5876 images from /kaggle/input/faceforensics/FaceForensic/train
Real: 2930, Fake: 2946
Loaded 395 images from /kaggle/input/faceforensics/FaceForensic/validation
Real: 198, Fake: 197
Loaded 199 images from /kaggle/input/faceforensics/FaceForensic/test
Real: 99, Fake: 100


Some weights of ViTModel were not initialized from the model checkpoint at google/vit-base-patch16-224 and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipykernel_237/3384088664.py:370: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()


...

  - Generating Denoised Image...
  ✓ Saved: research_visualizations/fake_sample_2.png

Processing: fake_721_715_frame290.png
  - Generating Activation Map...
  - Generating GradCAM...


/usr/local/lib/python3.12/dist-packages/torch/nn/modules/module.py:1864: FutureWarning: Using non-full backward hooks on a Module that does not take as input a single Tensor or a tuple of Tensors is deprecated and will be removed in future versions. This hook will be missing some of the grad_input. Please use register_full_backward_hook to get the documented behavior.
  self._maybe_warn_non_full_backward_hook(args, result, grad_fn)


  - Generating LIME explanation...


  0%|          | 0/300 [00:00<?, ?it/s]

  - Generating Denoised Image...
  ✓ Saved: research_visualizations/fake_sample_3.png


✓ ALL VISUALIZATIONS GENERATED SUCCESSFULLY!

Output directory: research_visualizations/
Total files created: 6

Files:
  - real_sample_1.png
  - real_sample_2.png
  - real_sample_3.png
  - fake_sample_1.png
  - fake_sample_2.png
  - fake_sample_3.png

Each visualization shows 5 methods horizontally for easy paper inclusion.

